<a href="https://colab.research.google.com/github/kej534923-maker/card-1995-iv-replication/blob/main/02_Replication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install linearmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 6.1 MB/s eta 0:00:00


In [2]:
import pandas as pd
import statsmodels.api as sm
from linearmodels.iv import IV2SLS

In [5]:
import re

def extract_varlist_from_sas(path="read1.sas"):
    text = open(path, "r", errors="ignore").read()

    m = re.search(r"(?mi)^\s*input\s+(.*?);", text, flags=re.DOTALL)
    if not m:
        raise ValueError("Cannot find INPUT block (line-start input ... ;)")

    block = m.group(1)

    block = re.sub(r"/\*.*?\*/", " ", block, flags=re.DOTALL)

    tokens = re.findall(r"\b[A-Za-z_][A-Za-z0-9_]*\b", block)

    return tokens

colnames = extract_varlist_from_sas("read1.sas")
print("Number of variables:", len(colnames))
print("First 10:", colnames[:10])
print("Last 5:", colnames[-5:])

Number of variables: 52
First 10: ['id', 'nearc2', 'nearc4', 'nearc4a', 'nearc4b', 'ed76', 'ed66', 'age76', 'daded', 'nodaded']
Last 5: ['iq', 'marsta76', 'marsta78', 'marsta80', 'libcrd14']


In [6]:
import pandas as pd

df = pd.read_csv(
    "nls.dat",
    sep=r"\s+",
    header=None,
    names=colnames,
    engine="python"
)

df["age76_sq"] = df["age76"] ** 2

print(df.shape)
df.head()

(3613, 53)


,id,nearc2,nearc4,nearc4a,nearc4b,ed76,ed66,age76,daded,nodaded,...,enroll76,enroll78,enroll80,kww,iq,marsta76,marsta78,marsta80,libcrd14,age76_sq
0,2,0,0,0,0,7,5,29,9.94,1,...,0,0,0,15,.,1,1,1,0,841
1,3,0,0,0,0,12,11,27,8.00,0,...,0,0,0,35,93,1,4,4,1,729
2,4,0,0,0,0,12,12,34,14.00,0,...,0,.,.,42,103,1,.,.,1,1156
3,5,1,1,1,0,11,11,27,11.00,0,...,0,.,0,25,88,1,.,5,1,729
4,6,1,1,1,0,12,12,34,8.00,0,...,0,0,.,34,108,1,1,.,0,1156


In [7]:
import numpy as np
import pandas as pd

# 把 '.' 当作缺失值
df = df.replace(".", np.nan)

# 把所有列都强制转成 numeric（转不了的变 NaN）
df = df.apply(pd.to_numeric, errors="coerce")

In [11]:
cols = ["lwage76", "ed76", "nearc4", "black", "south66", "smsa66r", "age76", "age76_sq"]

df_reg = df[cols].dropna().copy()

print(df_reg.shape)
print(df_reg.columns)

(3010, 8)
Index(['lwage76', 'ed76', 'nearc4', 'black', 'south66', 'smsa66r', 'age76',
       'age76_sq'],
      dtype='object')


In [12]:
X_first = df_reg[["nearc4", "black", "south66", "smsa66r", "age76", "age76_sq"]]
X_first = sm.add_constant(X_first)

first_stage = sm.OLS(df_reg["ed76"], X_first).fit()
print(first_stage.summary())

                            OLS Regression Results                            
Dep. Variable:                   ed76   R-squared:                       0.105
Model:                            OLS   Adj. R-squared:                  0.103
Method:                 Least Squares   F-statistic:                     58.59
Date:                Tue, 10 Mar 2026   Prob (F-statistic):           8.17e-69
Time:                        04:07:20   Log-Likelihood:                -7067.7
No. Observations:                3010   AIC:                         1.415e+04
Df Residuals:                    3003   BIC:                         1.419e+04
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -2.7629      4.336     -0.637      0.5

In [13]:
X = df_reg[["ed76", "black", "south66", "smsa66r", "age76", "age76_sq"]]
X = sm.add_constant(X)

y = df_reg["lwage76"]

ols_model = sm.OLS(y, X).fit()
print(ols_model.summary())

                            OLS Regression Results                            
Dep. Variable:                lwage76   R-squared:                       0.258
Model:                            OLS   Adj. R-squared:                  0.256
Method:                 Least Squares   F-statistic:                     174.0
Date:                Tue, 10 Mar 2026   Prob (F-statistic):          2.13e-190
Time:                        04:07:56   Log-Likelihood:                -1376.2
No. Observations:                3010   AIC:                             2766.
Df Residuals:                    3003   BIC:                             2808.
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          2.9490      0.654      4.506      0.0

In [15]:
df_reg["ed_hat"] = first_stage.predict(X_first)

df_reg[["ed76", "ed_hat"]].head()

,ed76,ed_hat
0,7,12.459351
1,12,13.801701
2,12,13.139710
3,11,14.241825
4,12,13.579834


In [16]:
X_second = df_reg[["ed_hat", "black", "south66", "smsa66r", "age76", "age76_sq"]]
X_second = sm.add_constant(X_second)

manual_2sls = sm.OLS(df_reg["lwage76"], X_second).fit()

print(manual_2sls.summary())

                            OLS Regression Results                            
Dep. Variable:                lwage76   R-squared:                       0.214
Model:                            OLS   Adj. R-squared:                  0.212
Method:                 Least Squares   F-statistic:                     136.1
Date:                Tue, 10 Mar 2026   Prob (F-statistic):          7.52e-153
Time:                        04:11:42   Log-Likelihood:                -1463.2
No. Observations:                3010   AIC:                             2940.
Df Residuals:                    3003   BIC:                             2983.
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          3.1156      0.682      4.570      0.0

In [18]:
from linearmodels.iv import IV2SLS

iv_model = IV2SLS.from_formula(
    "lwage76 ~ 1 + black + south66 + smsa66r + age76 + age76_sq + [ed76 ~ nearc4]",
    data=df_reg
).fit()

print(iv_model.summary)

                          IV-2SLS Estimation Summary                          
Dep. Variable:                lwage76   R-squared:                      0.1248
Estimator:                    IV-2SLS   Adj. R-squared:                 0.1230
No. Observations:                3010   F-statistic:                    789.68
Date:                Tue, Mar 10 2026   P-value (F-stat)                0.0000
Time:                        04:13:00   Distribution:                  chi2(6)
Cov. Estimator:                robust                                         
                                                                              
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
Intercept      3.1156     0.7206     4.3234     0.0000      1.7032      4.5280
black         -0.0904     0.0580    -1.5603     0.11

In [20]:
results_table = pd.DataFrame({
    "Model": ["OLS", "Manual 2SLS", "IV2SLS"],
    "Coefficient": [
        ols_model.params["ed76"],
        manual_2sls.params["ed_hat"],
        iv_model.params["ed76"]
    ],
    "Std_Error": [
        ols_model.bse["ed76"],
        manual_2sls.bse["ed_hat"],
        iv_model.std_errors["ed76"]
    ],
    "P_Value": [
        ols_model.pvalues["ed76"],
        manual_2sls.pvalues["ed_hat"],
        iv_model.pvalues["ed76"]
    ]
})

results_table

,Model,Coefficient,Std_Error,P_Value
0,OLS,0.037428,0.002748,4.766801e-41
1,Manual 2SLS,0.101209,0.040001,1.145203e-02
2,IV2SLS,0.101209,0.041779,1.541531e-02
